In [ ]:
import sys
print("Python Path:", sys.executable)
print("Looking good! Your environment is ready.")

In [ ]:
import os

# Dynamically resolve and inject the cudnn path into the environment
venv_base = os.path.expanduser("~/qchem/.venv/lib/python3.12/site-packages")
cudnn_path = os.path.join(venv_base, "nvidia", "cudnn", "lib")

if os.path.exists(cudnn_path):
    os.environ["LD_LIBRARY_PATH"] = cudnn_path + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"cuDNN Version: {torch.backends.cudnn.version()}")
print(f"CUDA Available (GPU): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
import torch

try:
    import torch_geometric
    print(f"PyTorch Geometric Version: {torch_geometric.__version__}")
    
    # Check for optional PyG acceleration extensions
    extensions = ['torch_scatter', 'torch_sparse', 'pyg_lib']
    print("\nExtension Status:")
    for ext in extensions:
        try:
            __import__(ext)
            print(f"  - {ext}: Installed")
        except ImportError:
            print(f"  - {ext}: NOT Installed (Optional)")
            
except ImportError:
    print("PyTorch Geometric is NOT installed in this environment.")

In [ ]:
try:
    import rdkit
    from rdkit import Chem
    from rdkit.Chem import Draw
    
    print(f"RDKit Version: {rdkit.__version__}")
    
    # Quick sanity check: Parse a simple SMILES string (Water)
    mol = Chem.MolFromSmiles('O')
    print(f"Sanity Check: Created molecule with {mol.GetNumAtoms()} atom(s) successfully.")
    
except ImportError:
    print("RDKit is NOT installed in this environment.")

In [ ]:
from cosmosis.dataset import ExampleDataset
ds = ExampleDataset().load_data("bang")
print(f"Example dataset: {ds[1]}") 

In [ ]:
for lib in ["transformers", "accelerate", "peft", "flash_attn"]:
    try:
        mod = __import__(lib)
        print(f"✅ {lib.replace('_', '-'):<13} installed: {mod.__version__}")
    except ImportError:
        print(f"❌ {lib.replace('_', '-'):<13} missing or broken")

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)

lora_config = LoraConfig(
    r=16,                                 
    lora_alpha=32,                          
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05,                      
    bias="none",                             
    task_type=TaskType.CAUSAL_LM             
)

lora_model = get_peft_model(base_model, lora_config)
model_device = next(lora_model.parameters()).device

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

prompts = [
    "What are the first 3 steps to bake cookies?",
    "Explain quantum computing in one sentence.",
    "Give me a short slogan for an eco-friendly water bottle."
]

for i, prompt_text in enumerate(prompts, start=1):
    print(f"\n--- Inference Pass {i} ---")
    print(f"Prompt: {prompt_text}")
    
    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    
    input_ids = inputs["input_ids"].to(model_device)
    attention_mask = inputs["attention_mask"].to(model_device)
    
    with torch.inference_mode():
        outputs = lora_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=50,      
            do_sample=True,        
            temperature=0.6,
            top_p=0.9,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_tokens = outputs[0, input_ids.shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    print(f"Response: {response}")
